# Ensemble Methods — Implementations

One extra lane per algorithm, all against sklearn. There is **no torch lane anywhere in this topic**: every model here is trees all the way down, and a tree is fit by a discrete argmax over candidate splits — its predictions are piecewise constant, the training objective has zero gradient almost everywhere and none at the thresholds, so autograd has nothing to act on. Two of the three fixtures match sklearn to float rounding because the algorithms are deterministic; the random forest cannot — scratch and sklearn draw bootstraps from different rng streams — so its lane compares only vote-level invariants.

## 06_decision_tree_regressor

Greedy MSE splitting; every leaf predicts the mean of what lands in it.

### library

sklearn's `DecisionTreeRegressor(criterion='squared_error')` runs the notebook's exact recipe: sort each feature, scan the midpoints, keep the split with the largest weighted-MSE decrease, recurse. On exact plateaus both learners recover the same partition, so predictions agree to the last bit. **What the library adds:** a Cython splitter, cost-complexity pruning and `min_*` regularisers, and an inspectable `tree_` — the criterion itself is `np.var`, as the checks confirm.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

# hints:
# 1. DecisionTreeRegressor(criterion='squared_error') is the notebook's MSE gain, verbatim.
# 2. sklearn sends x <= threshold left, the scratch tree x < — midpoints make it moot.
# 3. sklearn casts X to float32 internally, so thresholds match the midpoints only to ~1e-7.
# 4. tree_.impurity[0] equals np.var(y) after fit — the criterion is inspectable.


class ScratchDecisionTreeRegressor:
    """sklearn's CART regressor behind the notebook's interface.

    Same greedy recipe as `_best_split`/`_build`: for every feature, sort,
    scan the midpoints between consecutive distinct values, keep the split
    with the largest weighted-MSE decrease, recurse until the depth limit or
    a pure node. On exact plateaus both learners recover the same partition,
    and identical partitions mean identical leaf means."""

    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        self._model = DecisionTreeRegressor(
            criterion="squared_error", max_depth=self.max_depth,
            min_samples_split=self.min_samples_split, random_state=0)
        self._model.fit(np.asarray(X, dtype=float), np.asarray(y, dtype=float))
        return self

    def predict(self, X):
        return self._model.predict(np.asarray(X, dtype=float))


In [ ]:
# exports: tree_probe_pred, tree_train_mse
_rng_eq = np.random.default_rng(606)
X_tree_eq = _rng_eq.uniform(0.0, 1.0, size=(60, 1))
y_tree_eq = np.where(X_tree_eq[:, 0] < 0.3, 1.0,
                     np.where(X_tree_eq[:, 0] < 0.6, -2.0, 4.0))
X_probe_eq = np.array([[0.05], [0.15], [0.25], [0.4], [0.5], [0.65], [0.8], [0.95]])

_fit_eq = ScratchDecisionTreeRegressor(max_depth=3).fit(X_tree_eq, y_tree_eq)
tree_probe_pred = _fit_eq.predict(X_probe_eq)
tree_train_mse = float(np.mean((y_tree_eq - _fit_eq.predict(X_tree_eq)) ** 2))
print("probe predictions:", tree_probe_pred)
print("train MSE:", tree_train_mse, "| depth:", _fit_eq._model.get_depth(),
      "| leaves:", _fit_eq._model.get_n_leaves())


In [ ]:
assert tree_train_mse == 0.0, "three exact plateaus are fit exactly"
assert _fit_eq._model.get_depth() == 2, "two greedy splits suffice for three plateaus"

# The impurity sklearn reports at the root is the notebook's _mse, verbatim.
assert abs(_fit_eq._model.tree_.impurity[0] - np.var(y_tree_eq)) < 1e-12

# Trees compare, never measure: a monotone map of the feature changes nothing.
_fit_mono = ScratchDecisionTreeRegressor(max_depth=3).fit(2.0 * X_tree_eq + 1.0, y_tree_eq)
assert np.array_equal(_fit_mono.predict(2.0 * X_probe_eq + 1.0), tree_probe_pred), \
    "predictions are invariant under monotone feature transforms"

# A depth-1 stump gets one split: two leaves, and it must leave real error behind.
_stump = ScratchDecisionTreeRegressor(max_depth=1).fit(X_tree_eq, y_tree_eq)
_stump_pred = _stump.predict(X_tree_eq)
assert len(set(_stump_pred.tolist())) == 2, "a stump predicts exactly two values"
assert float(np.mean((y_tree_eq - _stump_pred) ** 2)) > 1.0, "one split cannot fit three plateaus"


## 06_random_forest

Bootstrapped rows, subsampled features, and a majority vote over trees.

### library

sklearn's `RandomForestClassifier` draws its bootstraps and feature subsets from a different rng stream than the scratch forest, so per-tree equality is impossible and is not claimed. The fixture separates the classes by a guaranteed gap, making the ensemble *vote* invariant, and this lane exports exactly those invariants: probe predictions, train accuracy, OOB accuracy. **What the library adds:** parallel fitting, soft voting via averaged `predict_proba` (identical to the scratch hard vote when leaves are pure), and OOB accounting built into `fit`.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# hints:
# 1. max_features='sqrt' is the same floor(sqrt(d)) rule as the scratch _feature_count.
# 2. Different rng streams draw the bootstraps — per-tree equality is impossible.
# 3. sklearn averages predict_proba (soft vote); the scratch forest counts hard votes.
# 4. With pure leaves each tree votes 0 or 1, so soft and hard voting coincide exactly.
# 5. oob_score=True makes fit() score each sample with the trees that never saw it.


class ScratchRandomForestClassifier:
    """sklearn's random forest behind the notebook's interface.

    Same ensemble recipe — bootstrap the rows, subsample the features at every
    split, aggregate by vote — but the bootstraps come from sklearn's own rng
    stream, so individual trees never match the scratch ones. Only vote-level
    quantities are comparable, and only on data where the vote is invariant."""

    def __init__(self, n_estimators=100, max_depth=None, max_features="sqrt",
                 random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state

    def _feature_count(self, n_features):
        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))
        if isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        return n_features

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self._model = RandomForestClassifier(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            max_features=self._feature_count(X.shape[1]),
            random_state=self.random_state, bootstrap=True, oob_score=True)
        self._model.fit(X, np.asarray(y))
        self.classes_ = self._model.classes_
        return self

    def predict(self, X):
        return self._model.predict(np.asarray(X, dtype=float))

    def score(self, X, y):
        return float(np.mean(self.predict(X) == np.asarray(y)))

    def oob_score(self, X, y):
        """sklearn computes OOB accuracy on the training set during fit; this
        returns that stored value, so call it with the training data as the
        scratch API does."""
        return float(self._model.oob_score_)


In [ ]:
# exports: rf_probe_pred, rf_train_acc, rf_oob_acc
_rng_eq = np.random.default_rng(88)
_centers_eq = np.repeat(np.array([[-3.0, -3.0], [3.0, 3.0]]), [42, 38], axis=0)
X_rf_eq = _centers_eq + _rng_eq.uniform(-1.2, 1.2, size=(80, 2))
y_rf_eq = np.repeat(np.array([0, 1]), [42, 38])
X_probe_rf_eq = np.array([[-3.0, -3.0], [-4.0, -2.0], [-2.5, -3.5],
                          [3.0, 3.0], [2.0, 4.0], [3.5, 2.5]])

_rf_eq = ScratchRandomForestClassifier(n_estimators=25, max_depth=None,
                                       random_state=3).fit(X_rf_eq, y_rf_eq)
rf_probe_pred = _rf_eq.predict(X_probe_rf_eq).astype(int)
rf_train_acc = _rf_eq.score(X_rf_eq, y_rf_eq)
rf_oob_acc = _rf_eq.oob_score(X_rf_eq, y_rf_eq)
print("probe votes:", rf_probe_pred, "| train acc:", rf_train_acc, "| OOB acc:", rf_oob_acc)


In [ ]:
assert rf_train_acc == 1.0 and rf_oob_acc == 1.0, "a guaranteed gap makes both accuracies exact"

# Soft voting (averaged probabilities) equals the scratch hard vote when every
# leaf is pure: each tree then votes exactly 0 or 1.
_votes = np.array([_t.predict(X_rf_eq.astype(np.float32)) for _t in _rf_eq._model.estimators_])
_hard = (_votes.mean(axis=0) > 0.5).astype(int)
assert np.array_equal(_hard, _rf_eq.predict(X_rf_eq)), "soft and hard voting agree here"

# Bagging really bagged: all 25 trees were fit on distinct bootstrap samples.
assert len({tuple(sorted(_s)) for _s in _rf_eq._model.estimators_samples_}) == 25

# Deep inside a blob the vote is unanimous, not merely majority.
assert float(_rf_eq._model.predict_proba(X_probe_rf_eq).max(axis=1).min()) == 1.0


## 06_gradient_boosting

Fit the residuals, shrink, add, repeat.

### library

sklearn's `GradientBoostingRegressor(loss='squared_error')` is the scratch loop verbatim: initialise with the mean, fit a depth-limited tree to the residuals, add it scaled by the learning rate. With no subsampling the fit is deterministic, so predictions and the entire loss curve match the scratch lane to ~1e-16. **What the library adds:** row subsampling, early stopping, `staged_predict`, and losses (huber, quantile) whose pseudo-residuals are no longer just `y − F`.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

# hints:
# 1. loss='squared_error' initialises with the mean — the scratch init, via DummyRegressor.
# 2. No subsample and a fixed max_depth: the fit is deterministic, like the scratch loop.
# 3. staged_predict replays the ensemble round by round — train_losses_ for free.
# 4. learning_rate scales every tree's contribution; leaves hold plain residual means.


class ScratchGradientBoostingRegressor:
    """sklearn's gradient boosting behind the notebook's interface.

    The identical loop: F0 = mean(y), then each round fits a depth-limited
    regression tree to the residuals y - F and adds it scaled by the learning
    rate. Squared loss makes every step deterministic, so this lane matches
    the scratch predictions and the entire loss curve to float rounding."""

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.initial_prediction_ = None
        self.train_losses_ = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self._model = GradientBoostingRegressor(
            n_estimators=self.n_estimators, learning_rate=self.learning_rate,
            max_depth=self.max_depth, loss="squared_error", random_state=0)
        self._model.fit(X, y)
        self.initial_prediction_ = float(self._model.init_.predict(X[:1]).ravel()[0])
        self.train_losses_ = [float(np.mean((y - F) ** 2))
                              for F in self._model.staged_predict(X)]
        return self

    def predict(self, X):
        return self._model.predict(np.asarray(X, dtype=float))


In [ ]:
# exports: gb_probe_pred, gb_losses, gb_init, gb_train_mse
_rng_eq = np.random.default_rng(909)
X_gb_eq = _rng_eq.uniform(-2.0, 2.0, size=(70, 2))
y_gb_eq = np.sin(X_gb_eq[:, 0]) + 0.5 * X_gb_eq[:, 1] ** 2 + 0.1 * _rng_eq.normal(size=70)
X_probe_gb_eq = np.array([[-1.5, -1.0], [-0.5, 0.5], [0.0, 0.0], [0.75, -0.25], [1.5, 1.0]])

_gb_eq = ScratchGradientBoostingRegressor(n_estimators=25, learning_rate=0.1,
                                          max_depth=2).fit(X_gb_eq, y_gb_eq)
gb_probe_pred = _gb_eq.predict(X_probe_gb_eq)
gb_losses = _gb_eq.train_losses_
gb_init = _gb_eq.initial_prediction_
gb_train_mse = float(np.mean((y_gb_eq - _gb_eq.predict(X_gb_eq)) ** 2))
print("init:", gb_init, "| final train MSE:", gb_train_mse)
print("probe predictions:", gb_probe_pred)


In [ ]:
assert abs(gb_init - float(np.mean(y_gb_eq))) < 1e-12, "squared-error init is the mean of y"
assert all(b < a for a, b in zip(gb_losses, gb_losses[1:])), \
    "every round strictly reduces the training MSE on this surface"
assert gb_losses[0] < float(np.mean((y_gb_eq - np.mean(y_gb_eq)) ** 2)), \
    "one round already beats the constant predictor"

# Shrinkage is real: a 10x smaller learning rate fits far less at the same budget.
_gb_slow = ScratchGradientBoostingRegressor(n_estimators=25, learning_rate=0.01,
                                            max_depth=2).fit(X_gb_eq, y_gb_eq)
assert _gb_slow.train_losses_[-1] > gb_train_mse, "lr=0.01 lags lr=0.1 after 25 rounds"
assert gb_train_mse < 0.1, "25 shrunken depth-2 trees fit the smooth surface well"
